# Aurora visibility — live globe

In [1]:
import datetime

import aacgmv2
import astral.moon
import joblib
import matplotlib
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from astral import Observer
from astral.sun import elevation as sun_elevation_fn

from aurora_transforms import cosine_mlon, sine_mlon  # required for joblib unpickling
from aurora_gated_model import GatedAuroraModel  # required for joblib unpickling
from fetch_live_geomagnetic import get_live_geomagnetic_indices

MODEL_PATH = "aurora_ebm_model_final_deployment.joblib"

In [2]:
dt = datetime.datetime.now(datetime.timezone.utc)

geo_lats = np.arange(30, 91, 1, dtype=float)
geo_lons = np.arange(-180, 181, 2, dtype=float)
geo_lon_mesh, geo_lat_mesh = np.meshgrid(geo_lons, geo_lats)
geo_lat_flat = geo_lat_mesh.flatten()
geo_lon_flat = geo_lon_mesh.flatten()

geo = get_live_geomagnetic_indices()

mlat_flat, mlon_flat, _mlt_flat = aacgmv2.get_aacgm_coord_arr(
    geo_lat_flat, geo_lon_flat, np.zeros_like(geo_lat_flat), dt.replace(tzinfo=None)
)
oval_dist_flat = np.abs(mlat_flat) - (66.0 - 2.0 * geo["kp"])

sun_elev_flat = np.array([
    sun_elevation_fn(Observer(latitude=la, longitude=lo), dt)
    for la, lo in zip(geo_lat_flat, geo_lon_flat)
])

moon_darkness = astral.moon.phase(dt.date()) / 28.0
moon_darkness = 1 - 2 * min(moon_darkness, 1 - moon_darkness)

In [3]:
sweep_df = pd.DataFrame({
    "mlat": mlat_flat, "mlon": mlon_flat, "kp": geo["kp"], "bz_gsm": geo["bz_gsm"],
    "solar_wind_speed": geo["solar_wind_speed"], "oval_dist": oval_dist_flat,
    "Ap": geo["ap_kp_implied"], "cloud_cover": 0.0, "moon_darkness": moon_darkness,
    "sun_elevation_utc": sun_elev_flat,
})
sweep_df["geo_lat"] = geo_lat_flat
sweep_df["geo_lon"] = geo_lon_flat

bundle = joblib.load(MODEL_PATH)
valid = sweep_df[bundle["featureslow"]].notna().all(axis=1)
sweep_df["probability"] = np.nan
sweep_df.loc[valid, "probability"] = bundle["pipeline"].predict_proba(sweep_df.loc[valid, bundle["featureslow"]])[:, 1]

In [4]:
cmap = matplotlib.colormaps.get_cmap("YlGn")
plot_df = sweep_df.dropna(subset=["probability"]).copy()


def darkness_opacity(sun_elev):
    return float(np.clip(1 - sun_elev / 30, 0.05, 1.0))


colors, hover_text = [], []
for _, row in plot_df.iterrows():
    r, g, b, _ = cmap(row["probability"])
    a = darkness_opacity(row["sun_elevation_utc"])
    colors.append(f"rgba({r*255:.0f},{g*255:.0f},{b*255:.0f},{a:.2f})")
    hover_text.append(
        f"P(aurora): {row['probability']:.1%}<br>mlat: {row['mlat']:.1f}\u00b0<br>"
        f"sun elevation: {row['sun_elevation_utc']:.1f}\u00b0<br>kp: {row['kp']:.2f}"
    )

fig = go.Figure()
fig.add_trace(go.Scattergeo(
    lat=plot_df["geo_lat"], lon=plot_df["geo_lon"], mode="markers",
    marker=dict(size=5, color=colors, line=dict(width=0)),
    text=hover_text, hoverinfo="text", showlegend=False,
))
fig.add_trace(go.Scattergeo(
    lat=[None], lon=[None],
    marker=dict(color=[0, 1], colorscale="YlGn", cmin=0, cmax=1, colorbar=dict(title="P(aurora)", len=0.6), size=0),
    showlegend=False,
))
fig.update_geos(
    projection_type="orthographic", projection_rotation=dict(lon=0, lat=90, roll=0),
    showland=True, landcolor="rgb(235,235,235)", showocean=True, oceancolor="rgb(205,228,247)",
    showcountries=True, countrycolor="rgb(160,160,160)", showcoastlines=True, coastlinecolor="rgb(120,120,120)",
)
fig.update_layout(
    title=dict(text=(
        f"Live aurora visibility \u2014 {dt.strftime('%Y-%m-%d %H:%M UTC')}"
        "<br><sup>\u26a0\ufe0f cloud_cover fixed at 0 (clear sky) for every point, no live weather info at every point </sup>"
    )),
    height=750, margin=dict(l=0, r=0, t=80, b=0),
)
fig.write_html("live_aurora_map.html")
fig.show()